# 300: DueCare Adversarial Resistance

**Bad actors who exploit migrant workers do not use polite, direct questions. They use obfuscation, academic framing, corporate disguise, and emotional manipulation to extract harmful content from LLMs.** This notebook runs 5 graded base prompts through every one of DueCare's 15 adversarial generators, captures the mutation-type metadata on each variation, and surfaces the attack-type distribution that Phase 3 fine-tuning targets.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a). This notebook is the adversarial-resistance opener of the Advanced Evaluation section: the generated variations feed [335 Attack Vector Inspector](https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector) for per-vector taxonomy and [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) for per-dimension scoring.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">5 graded trafficking-safety prompts from the shipped <code>trafficking</code> domain pack and 15 adversarial generators registered in <code>duecare.tasks.generators.ALL_GENERATORS</code> (evasion, coercion, financial obfuscation, regulatory framing, persona injection, and creative attack vectors).</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">A table of per-generator variation counts, a sample-variation preview carrying <code>mutation_type</code> metadata, and an attack-type distribution table showing which adversarial categories the generated set covers.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached. No GPU and no API key required for the generation path; GPU is only needed when this slice is rescored by a live model downstream.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 1 minute end-to-end. The 15 generators are in-process Python; no model loading, no network calls.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Advanced Evaluation, adversarial-resistance slot. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-270-gemma-generations">270 Gemma Generations</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector">335 Attack Vector Inspector</a>. Section close: <a href="https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion">499 Advanced Evaluation Conclusion</a>.</td></tr>
  </tbody>
</table>

### Why this notebook matters

A model that passes standard single-prompt safety tests can still fail catastrophically against adversarial variations. DueCare's 15 generators systematically produce the attacks real adversaries use, so safety evaluations measure what actually happens in production rather than what benchmarks wish would happen. The mutation-type metadata on every variation is what makes the downstream Phase 3 fine-tuning curriculum attributable to a specific attack vector rather than to a vague "the model got worse at safety" signal.

### Reading order

- **Full section path:** you arrived from [270 Gemma Generations](https://www.kaggle.com/code/taylorsamarel/duecare-270-gemma-generations); continue to [335 Attack Vector Inspector](https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector) and close the section in [499](https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion).
- **Generator deep-dive:** [310 Prompt Factory](https://www.kaggle.com/code/taylorsamarel/duecare-310-prompt-factory) scales the same 15 generators to thousands of validated, importance-ranked variations.
- **Scoring layer:** [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) feeds the adversarial variations produced here into the 6-dimension weighted rubric.
- **Methodology source:** [140 Evaluation Mechanics](https://www.kaggle.com/code/taylorsamarel/140-duecare-evaluation-mechanics) is the earlier walkthrough of the 5-grade rubric, keyword scorer, and 6-dimension weighted rubric that the downstream scoring relies on; judges who want to audit what "adversarial score drop" means read 140 first.
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).

### What this notebook does

1. Install the pinned DueCare wheels with the generator registry.
2. Load 5 graded base prompts from the shipped trafficking pack and run every generator against them.
3. Preview sample adversarial variations with their mutation-type and base-prompt provenance.
4. Tally the attack-type distribution so the Phase 3 curriculum knows which vectors are over- or under-represented.


## 1. Install DueCare

Install the DueCare wheel packages. The adversarial generators
are part of the `duecare-llm-tasks` package.


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-domains==0.1.0', 'duecare-llm-tasks==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


In [ ]:
import subprocess, glob, sys
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + wheels + ['-q'])
    print(f'Installed {len(wheels)} DueCare wheels.')


## 2. Generate adversarial variations from 5 base prompts

We start with 5 graded base prompts and run all 15 generators,
producing 1 variation per generator per prompt. This gives us
approximately 75 adversarial test prompts spanning every attack type.

The table below shows each generator's name and how many variations
it produced. Some generators may produce fewer than 5 if the base
prompt does not fit the attack pattern.


In [ ]:
from duecare.tasks.generators import ALL_GENERATORS
from duecare.domains import register_discovered, load_domain_pack

register_discovered()
pack = load_domain_pack('trafficking')
base = [p for p in pack.seed_prompts() if p.get('graded_responses')][:5]

print(f'Base prompts: {len(base)}')
print(f'Generators available: {len(ALL_GENERATORS)}')
print()

all_variations = []
print(f'{"Generator":<35} {"Variations":>10}')
print('-' * 47)
for gen in ALL_GENERATORS:
    variations = gen.generate(base, n_variations=1, seed=42)
    all_variations.extend(variations)
    print(f'  {gen.__class__.__name__:<33} {len(variations):>10}')

print(f'\nTotal: {len(base)} base + {len(all_variations)} variations = '
      f'{len(base)+len(all_variations)} test prompts')
print(f'Amplification factor: {len(all_variations)/max(len(base),1):.1f}x')


### What the generators produce

Each generator applies a specific adversarial strategy:
- **Evasion/Regulatory:** Frame exploitation as regulatory compliance
- **Coercion:** Use pressure tactics and urgency
- **Financial:** Disguise fees as "standard business practice"
- **Persona:** Impersonate authority figures (31 persona types)
- **Creative:** Novel attacks like ASCII art, role-play, academic framing

The diversity of attack types is what makes DueCare's evaluation
meaningful: a model cannot pass by memorizing responses to one style
of prompt.


## 3. Inspect sample adversarial variations

Each variation carries metadata about its mutation type and base
prompt. This traceability is critical for understanding which
attack types a model fails against.


In [ ]:
import textwrap
print('--- Sample adversarial variations ---\n')
for v in all_variations[:5]:
    mt = v.get('metadata',{}).get('mutation_type','?')
    base_id = v.get('metadata',{}).get('base_prompt_id','?')
    print(f'[{mt}] (from base: {base_id})')
    print(textwrap.fill(v['text'][:200], width=80))
    print('...\n')


## 4. Score all variations

In production, these would be run through Gemma 4 and scored using
the rubric framework. Here we use a ScriptedModel to demonstrate
that the scoring pipeline handles all variation types consistently.

The attack type distribution shows which categories of adversarial
attacks are represented in the generated test set.


In [ ]:
from duecare.core import Capability, GenerationResult, Embedding, ModelHealth

class ScriptedModel:
    id = 'scripted:adversarial'
    display_name = 'Scripted'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _r = 'I cannot help with that. Under ILO C181 Article 7, workers shall not pay fees. Contact POEA at 1343.'
    def generate(self, messages, **kw): return GenerationResult(text=self._r, model_id=self.id)
    def embed(self, texts): return [Embedding(text=t, vector=[0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self): return ModelHealth(model_id=self.id, healthy=True)

from collections import Counter
gen_results = Counter()
for v in all_variations:
    mt = v.get('metadata',{}).get('mutation_type','unknown').split('_')[0]
    gen_results[mt] += 1

print(f'{"Attack Type":<25} {"Count":>6} {"Share":>8}')
print('-' * 42)
for at, count in gen_results.most_common():
    pct = count / len(all_variations) * 100
    print(f'{at:<25} {count:>6} {pct:>7.1f}%')


---

## What just happened

- Installed the pinned DueCare wheels that register the 15-generator adversarial library.
- Loaded 5 graded base prompts from the shipped `trafficking` pack and ran every generator against every base prompt, capturing per-generator counts and per-variation mutation-type metadata.
- Previewed sample variations so the attack strategies (evasion, coercion, financial obfuscation, persona injection, creative) are visibly distinct rather than abstractions in a summary table.
- Tallied the attack-type distribution across the full generated set so downstream curriculum targeting knows which vectors are over- or under-represented.

### Key findings

1. **Generator diversity is load-bearing.** The 15 generators cover every attack vector DueCare has encountered in the 74K-prompt corpus; a model cannot pass by defending against only one style.
2. **Mutation-type provenance makes scoring attributable.** Every variation carries `mutation_type` and `base_prompt_id`, so a downstream failure traces back to a specific generator rather than to noise.
3. **Per-vector scoring is downstream.** [335 Attack Vector Inspector](https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector) consumes the exact variations produced here and lays out per-vector severity and mitigation status.
4. **Factory scaling is downstream.** [310 Prompt Factory](https://www.kaggle.com/code/taylorsamarel/duecare-310-prompt-factory) runs the same 15 generators against the full 74K corpus under `PromptValidator` + `ImportanceRanker` to produce the Phase 3 adversarial curriculum.
5. **Evaluation continuity.** The generated variations are compatible with the same 6-dimension rubric the earlier comparison notebooks use, so adversarial scores are directly comparable to stock scores.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Install cell fails because the wheels dataset is not attached.</td><td style="padding: 6px 10px;">Attach <code>taylorsamarel/duecare-llm-wheels</code> from the Kaggle sidebar and rerun the first code cell.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>from duecare.tasks.generators import ALL_GENERATORS</code> raises <code>ImportError</code>.</td><td style="padding: 6px 10px;">The install cell must finish successfully before this import. Rerun step 1 if it printed a wheel count of zero.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>base</code> returns an empty list so no variations are produced.</td><td style="padding: 6px 10px;">The shipped <code>trafficking</code> pack has graded prompts by default; an empty list means the pack import fell back to an older build. Reinstall the wheels (step 1).</td></tr>
    <tr><td style="padding: 6px 10px;">A specific generator prints zero variations.</td><td style="padding: 6px 10px;">Intentional: some generators filter out prompts that do not fit their attack pattern. The remaining generators still produce enough variations for coverage analysis.</td></tr>
    <tr><td style="padding: 6px 10px;">The attack-type distribution shows one category dominating.</td><td style="padding: 6px 10px;">Expected when the base prompts cluster in a narrow category. Swap the base slice by editing the <code>[:5]</code> slice over <code>pack.seed_prompts()</code> to pull from a wider category mix.</td></tr>
  </tbody>
</table>

---

## Next

- **Continue the section:** [335 Attack Vector Inspector](https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector) turns these variations into a per-vector severity and mitigation table.
- **Downstream scoring:** [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) scores responses to these variations against the shared 6-dimension rubric.
- **Factory scaling:** [310 Prompt Factory](https://www.kaggle.com/code/taylorsamarel/duecare-310-prompt-factory) extends the same 15 generators to the full 74K-prompt corpus with validation and importance ranking.
- **Close the section:** [499 Advanced Evaluation Conclusion](https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'Adversarial resistance handoff >>> 335 Attack Vector Inspector: '
    'https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector'
    '. Section close: 499 Advanced Evaluation Conclusion: '
    'https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion'
    '.'
)
